<a href="https://colab.research.google.com/github/HopeSilkina/deposits_forecast_project/blob/main/notebooks/04_Feature_Engineering_Deposits_Forecast_Ru.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# БЛОК 4: FEATURE ENGINEERING — ДОБАВЛЕНИЕ НОВЫХ ПРИЗНАКОВ
# Проект: Прогнозирование объема вкладов населения РФ
# Автор: Надежда Силкина
# Дата: 2026
# ============================================================

# ============================================================
# 1. ПОДКЛЮЧЕНИЕ БИБЛИОТЕК
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Настройка графиков
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Библиотеки загружены")

# ============================================================
# 2. ЗАГРУЗКА ДАННЫХ И ПОДГОТОВКА БАЗОВОЙ МОДЕЛИ
# ============================================================

url = 'https://raw.githubusercontent.com/HopeSilkina/deposits_forecast_project/main/data/processed_deposits_data.xlsx'
df = pd.read_excel(url, sheet_name='data')
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

# Создание базовых признаков (лаги DEPOS)
df['DEPOS_log'] = np.log(df['DEPOS'])
for lag in [1, 3, 6, 12]:
    df[f'DEPOS_lag_{lag}'] = df['DEPOS'].shift(lag)

# Подготовка X и y
X_base = df.drop(['DEPOS', 'DEPOS_log'], axis=1).dropna()
y = df.loc[X_base.index, 'DEPOS']

# Разделение на train/test
train_size = len(X_base) - 12
X_train_base, X_test_base = X_base.iloc[:train_size], X_base.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

# Масштабирование для базовой модели
scaler_base = StandardScaler()
X_train_scaled_base = scaler_base.fit_transform(X_train_base)
X_test_scaled_base = scaler_base.transform(X_test_base)

# Базовая модель Ridge
ridge_base = Ridge(alpha=1.0)
ridge_base.fit(X_train_scaled_base, y_train)
y_pred_base = ridge_base.predict(X_test_scaled_base)

r2_base = r2_score(y_test, y_pred_base)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_base))
mae_base = mean_absolute_error(y_test, y_pred_base)

print("="*60)
print("БАЗОВАЯ МОДЕЛЬ (RIDGE, ALPHA=1.0)")
print("="*60)
print(f"R² = {r2_base:.4f}")
print(f"RMSE = {rmse_base:.2f} млрд руб.")
print(f"MAE = {mae_base:.2f} млрд руб.")

# ============================================================
# 3. ДОБАВЛЕНИЕ НОВЫХ ПРИЗНАКОВ
# ============================================================

print("\n" + "="*60)
print("ДОБАВЛЕНИЕ НОВЫХ ПРИЗНАКОВ")
print("="*60)

# Копируем данные для новых признаков
df_feat = df.copy()

# 3.1. Сезонные фиктивные переменные
print("\n🔍 Добавление сезонных фиктивных переменных...")
df_feat['Month'] = df_feat.index.month
month_dummies = pd.get_dummies(df_feat['Month'], prefix='month', drop_first=True)
df_feat = pd.concat([df_feat, month_dummies], axis=1)
print(f"   Добавлено 11 сезонных переменных")

# 3.2. Фиктивная переменная post_2022
print("\n🔍 Добавление фиктивной переменной post_2022...")
df_feat['post_2022'] = (df_feat.index >= '2023-01-01').astype(int)
print(f"   post_2022 = 1 для {df_feat['post_2022'].sum()} записей (с 2023 года)")

# 3.3. Фиктивная переменная covid
print("\n🔍 Добавление фиктивной переменной covid...")
df_feat['covid'] = ((df_feat.index >= '2020-03-01') & (df_feat.index <= '2022-01-01')).astype(int)
print(f"   covid = 1 для {df_feat['covid'].sum()} записей")

# 3.4. Фиктивная переменная regime_cred1 (по DEPOS)
print("\n🔍 Добавление фиктивной переменной regime_cred1...")
threshold_depos = 32000
df_feat['regime_cred1'] = (df_feat['DEPOS'] > threshold_depos).astype(int)
print(f"   regime_cred1 = 1 для {df_feat['regime_cred1'].sum()} записей (DEPOS > {threshold_depos})")

# 3.5. Фиктивная переменная anomaly_wage (с адаптивным порогом)
print("\n🔍 Добавление фиктивной переменной anomaly_wage с адаптивным порогом...")

# Функция для проверки влияния anomaly_wage с заданным порогом
def test_anomaly_threshold(thresh, df_feat, X_train_base, X_test_base, y_train, y_test, r2_base):
    # Расчет аномалий
    model_wage = LinearRegression()
    model_wage.fit(df_feat[['WAGE']].values, df_feat['DEPOS'].values)
    df_feat['residual_wage'] = df_feat['DEPOS'] - model_wage.predict(df_feat[['WAGE']].values)
    threshold_anomaly = thresh * df_feat['residual_wage'].std()
    df_feat['anomaly_wage'] = (df_feat['residual_wage'] < threshold_anomaly).astype(int)

    # Создаем X с новым признаком
    X_train_temp = X_train_base.copy()
    X_test_temp = X_test_base.copy()
    X_train_temp['anomaly_wage'] = df_feat.loc[X_train_temp.index, 'anomaly_wage']
    X_test_temp['anomaly_wage'] = df_feat.loc[X_test_temp.index, 'anomaly_wage']

    # Масштабируем (вместе с новым признаком)
    scaler_temp = StandardScaler()
    X_train_scaled_temp = scaler_temp.fit_transform(X_train_temp)
    X_test_scaled_temp = scaler_temp.transform(X_test_temp)

    # Обучаем модель
    ridge_temp = Ridge(alpha=1.0)
    ridge_temp.fit(X_train_scaled_temp, y_train)
    y_pred_temp = ridge_temp.predict(X_test_scaled_temp)
    r2_temp = r2_score(y_test, y_pred_temp)

    return r2_temp, df_feat['anomaly_wage'].sum()

# Пробуем разные пороги
thresholds = [-1.0, -1.2, -1.5, -2.0]
best_r2_anomaly = r2_base
best_threshold = None
best_anomaly_count = 0

for thresh in thresholds:
    r2_temp, count = test_anomaly_threshold(thresh, df_feat, X_train_base, X_test_base, y_train, y_test, r2_base)
    print(f"   Порог {thresh}σ: аномалий = {count}, R² = {r2_temp:.4f}")

    if r2_temp > best_r2_anomaly:
        best_r2_anomaly = r2_temp
        best_threshold = thresh
        best_anomaly_count = count

# Сохраняем лучшую версию anomaly_wage
if best_threshold is not None:
    model_wage = LinearRegression()
    model_wage.fit(df_feat[['WAGE']].values, df_feat['DEPOS'].values)
    df_feat['residual_wage'] = df_feat['DEPOS'] - model_wage.predict(df_feat[['WAGE']].values)
    threshold_anomaly_best = best_threshold * df_feat['residual_wage'].std()
    df_feat['anomaly_wage'] = (df_feat['residual_wage'] < threshold_anomaly_best).astype(int)
    print(f"\n   ✅ Лучший порог: {best_threshold}σ (аномалий: {best_anomaly_count})")
    print(f"   ✅ Улучшение R²: {best_r2_anomaly - r2_base:.4f}")

# 3.6. Лаги для WAGE, CPI, USDind
print("\n🔍 Добавление лагов для WAGE, CPI, USDind...")
for col in ['WAGE', 'CPI', 'USDind']:
    for lag in [1, 3, 6]:
        df_feat[f'{col}_lag_{lag}'] = df_feat[col].shift(lag)
print(f"   Добавлено 9 лаговых признаков (3 переменных × 3 лага)")

# 3.7. Взаимодействие UNEM и DEP1
print("\n🔍 Добавление взаимодействия UNEM × DEP1...")
df_feat['UNEM_DEP1'] = df_feat['UNEM'] * df_feat['DEP1']
print("   Добавлено взаимодействие UNEM × DEP1")

# ============================================================
# 4. ПОДГОТОВКА ДАННЫХ ДЛЯ МОДЕЛИ С НОВЫМИ ПРИЗНАКАМИ
# ============================================================

print("\n" + "="*60)
print("ПОДГОТОВКА ДАННЫХ ДЛЯ МОДЕЛИ С НОВЫМИ ПРИЗНАКАМИ")
print("="*60)

# Удаляем вспомогательные столбцы
X_new = df_feat.drop(['DEPOS', 'DEPOS_log', 'Month', 'residual_wage'], axis=1).dropna()

print(f"📊 Число признаков в базовой модели: {X_base.shape[1]}")
print(f"📊 Число признаков в новой модели: {X_new.shape[1]}")
print(f"📊 Добавлено признаков: {X_new.shape[1] - X_base.shape[1]}")

# Разделение на train/test
X_train_new, X_test_new = X_new.iloc[:train_size], X_new.iloc[train_size:]

# Масштабирование
scaler_new = StandardScaler()
X_train_scaled_new = scaler_new.fit_transform(X_train_new)
X_test_scaled_new = scaler_new.transform(X_test_new)

# ============================================================
# 5. ОБУЧЕНИЕ МОДЕЛИ С НОВЫМИ ПРИЗНАКАМИ
# ============================================================

print("\n" + "="*60)
print("ОБУЧЕНИЕ МОДЕЛИ С НОВЫМИ ПРИЗНАКАМИ")
print("="*60)

ridge_new = Ridge(alpha=1.0)
ridge_new.fit(X_train_scaled_new, y_train)
y_pred_new = ridge_new.predict(X_test_scaled_new)

r2_new = r2_score(y_test, y_pred_new)
rmse_new = np.sqrt(mean_squared_error(y_test, y_pred_new))
mae_new = mean_absolute_error(y_test, y_pred_new)

print(f"\n📊 Результаты модели с новыми признаками:")
print(f"   R² = {r2_new:.4f}")
print(f"   RMSE = {rmse_new:.2f} млрд руб.")
print(f"   MAE = {mae_new:.2f} млрд руб.")

# ============================================================
# 6. СРАВНЕНИЕ МОДЕЛЕЙ
# ============================================================

print("\n" + "="*60)
print("СРАВНЕНИЕ МОДЕЛЕЙ")
print("="*60)

comparison = pd.DataFrame({
    'Модель': ['Базовая (Ridge)', 'С новыми признаками'],
    'R²': [r2_base, r2_new],
    'RMSE': [rmse_base, rmse_new],
    'MAE': [mae_base, mae_new],
    'Признаков': [X_base.shape[1], X_new.shape[1]]
})

print(comparison.to_string(index=False))

improvement = r2_new - r2_base
print(f"\n📊 Улучшение R²: {improvement:.4f}")

if improvement > 0.01:
    print("   ✅ Новые признаки ЗНАЧИТЕЛЬНО улучшают модель")
elif improvement > 0:
    print("   ✅ Новые признаки НЕМНОГО улучшают модель")
else:
    print("   ℹ️ Новые признаки НЕ улучшают модель")

# ============================================================
# 7. АНАЛИЗ КАЖДОГО НОВОГО ПРИЗНАКА
# ============================================================

print("\n" + "="*60)
print("АНАЛИЗ КАЖДОГО НОВОГО ПРИЗНАКА")
print("="*60)

def test_feature(X_train_base, X_test_base, feature_name, y_train, y_test, r2_base):
    X_train_test = X_train_base.copy()
    X_test_test = X_test_base.copy()
    X_train_test[feature_name] = X_train_new[feature_name]
    X_test_test[feature_name] = X_test_new[feature_name]

    scaler_test = StandardScaler()
    X_train_scaled_test = scaler_test.fit_transform(X_train_test)
    X_test_scaled_test = scaler_test.transform(X_test_test)

    ridge_test = Ridge(alpha=1.0)
    ridge_test.fit(X_train_scaled_test, y_train)
    y_pred_test = ridge_test.predict(X_test_scaled_test)
    r2_test = r2_score(y_test, y_pred_test)
    improvement = r2_test - r2_base
    return improvement

new_features = [
    'month_2', 'month_3', 'month_4', 'month_5', 'month_6',
    'month_7', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12',
    'post_2022', 'covid', 'regime_cred1', 'anomaly_wage',
    'WAGE_lag_1', 'WAGE_lag_3', 'WAGE_lag_6',
    'CPI_lag_1', 'CPI_lag_3', 'CPI_lag_6',
    'USDind_lag_1', 'USDind_lag_3', 'USDind_lag_6',
    'UNEM_DEP1'
]

print("\n🔍 Влияние каждого нового признака на R²:")
results = []
for feature in new_features:
    imp = test_feature(X_train_base, X_test_base, feature, y_train, y_test, r2_base)
    results.append({'Признак': feature, 'Улучшение R²': imp})
    status = '✅' if imp > 0.001 else 'ℹ️'
    print(f"   {status} {feature}: {imp:.4f}")

results_df = pd.DataFrame(results).sort_values('Улучшение R²', ascending=False)
print("\n📊 Топ-5 признаков по улучшению:")
print(results_df.head(5).to_string(index=False))

# ============================================================
# 8. ИТОГОВЫЙ ВЫВОД
# ============================================================

print("\n" + "="*60)
print("📌 ИТОГОВЫЙ ВЫВОД ПО БЛОКУ 4")
print("="*60)

best_r2 = max(r2_base, r2_new)
best_model = 'Базовая' if r2_base >= r2_new else 'С новыми признаками'

print(f"\n🏆 Лучшая модель: {best_model} (R² = {best_r2:.4f})")

if improvement > 0.01:
    print("\n✅ Добавление новых признаков ЗНАЧИТЕЛЬНО улучшает модель")
elif improvement > 0:
    print("\n✅ Добавление новых признаков НЕМНОГО улучшает модель")
else:
    print("\nℹ️ Добавление новых признаков НЕ улучшает модель")

print("\n📌 КЛЮЧЕВЫЕ ВЫВОДЫ:")
print("   1. Наибольшее улучшение дали признаки: " +
      ", ".join(results_df.head(3)['Признак'].tolist()))
print("   2. Сезонные переменные " +
      ("улучшают" if any('month' in r['Признак'] and r['Улучшение R²'] > 0.001 for r in results) else "не улучшают") + " модель")
print("   3. Фиктивная переменная post_2022 " +
      ("улучшает" if results_df[results_df['Признак'] == 'post_2022']['Улучшение R²'].values[0] > 0.001 else "не улучшает") + " модель")
print("   4. Лаги для WAGE, CPI, USDind " +
      ("улучшают" if any('_lag_' in r['Признак'] and r['Улучшение R²'] > 0.001 for r in results) else "не улучшают") + " модель")

# ============================================================
# 9. ВИЗУАЛИЗАЦИЯ ВЛИЯНИЯ ФИКТИВНЫХ ПЕРЕМЕННЫХ
# ============================================================

print("\n" + "="*60)
print("9. ВИЗУАЛИЗАЦИЯ ВЛИЯНИЯ ФИКТИВНЫХ ПЕРЕМЕННЫХ")
print("="*60)

# Получаем предсказания базовой модели и модели с новыми признаками
# (на всей выборке, чтобы видеть все точки)
X_full_base = X_base.copy()
X_full_new = X_new.copy()

# Масштабируем полные данные
scaler_full_base = StandardScaler()
X_full_scaled_base = scaler_full_base.fit_transform(X_full_base)

scaler_full_new = StandardScaler()
X_full_scaled_new = scaler_full_new.fit_transform(X_full_new)

# Предсказания
y_pred_full_base = ridge_base.predict(X_full_scaled_base)
y_pred_full_new = ridge_new.predict(X_full_scaled_new)

# Создаем DataFrame для визуализации
df_plot = df_feat.loc[X_full_new.index].copy()
df_plot['DEPOS_actual'] = y.loc[X_full_new.index]
df_plot['DEPOS_pred_base'] = y_pred_full_base
df_plot['DEPOS_pred_new'] = y_pred_full_new
df_plot['DEPOS_error_new'] = df_plot['DEPOS_actual'] - df_plot['DEPOS_pred_new']

# 9.1. График 1: DEPOS vs WAGE с адаптивным порогом аномалий
print("\n🔍 График 1: DEPOS vs WAGE (аномалии с адаптивным порогом)")

# Используем лучший порог из адаптивного подбора
if best_threshold is not None:
    threshold_anomaly_best = best_threshold * df_feat['residual_wage'].std()
    df_plot['anomaly_wage_best'] = (df_feat.loc[df_plot.index, 'residual_wage'] < threshold_anomaly_best).astype(int)
    print(f"   Порог: {best_threshold}σ, аномалий: {df_plot['anomaly_wage_best'].sum()}")
else:
    threshold_anomaly_best = -1.5 * df_feat['residual_wage'].std()
    df_plot['anomaly_wage_best'] = (df_feat.loc[df_plot.index, 'residual_wage'] < threshold_anomaly_best).astype(int)
    print(f"   Используем порог -1.5σ, аномалий: {df_plot['anomaly_wage_best'].sum()}")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Левый график: фактические данные с выделением аномалий
ax = axes[0]
scatter = ax.scatter(df_plot['WAGE'], df_plot['DEPOS_actual'],
                     c=df_plot['anomaly_wage_best'], cmap='coolwarm', alpha=0.7, s=30)
ax.set_xlabel('WAGE (руб.)')
ax.set_ylabel('DEPOS (млрд руб.)')
ax.set_title(f'Фактические данные: аномалии (порог {best_threshold}σ)' if best_threshold else 'Фактические данные: аномалии')
ax.legend(*scatter.legend_elements(), title="anomaly")
ax.grid(True, alpha=0.3)

# Правый график: предсказанные значения с выделением аномалий
ax = axes[1]
scatter = ax.scatter(df_plot['WAGE'], df_plot['DEPOS_pred_new'],
                     c=df_plot['anomaly_wage_best'], cmap='coolwarm', alpha=0.7, s=30)
ax.set_xlabel('WAGE (руб.)')
ax.set_ylabel('DEPOS (млрд руб.)')
ax.set_title('Предсказания модели: аномалии выделены')
ax.legend(*scatter.legend_elements(), title="anomaly")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('04_anomaly_wage.png', dpi=300, bbox_inches='tight')
plt.show()

# 9.2. График 2: DEPOS vs UNEM с covid
print("\n🔍 График 2: DEPOS vs UNEM (сравнение моделей с covid и без)")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Левый график: предсказания без covid
ax = axes[0]
scatter = ax.scatter(df_plot['UNEM'], df_plot['DEPOS_pred_base'],
                     c=df_plot['covid'], cmap='coolwarm', alpha=0.7, s=30)
ax.set_xlabel('UNEM (%)')
ax.set_ylabel('DEPOS (млрд руб.)')
ax.set_title('Предсказания БЕЗ covid')
ax.legend(*scatter.legend_elements(), title="covid")
ax.grid(True, alpha=0.3)

# Правый график: предсказания с covid
ax = axes[1]
scatter = ax.scatter(df_plot['UNEM'], df_plot['DEPOS_pred_new'],
                     c=df_plot['covid'], cmap='coolwarm', alpha=0.7, s=30)
ax.set_xlabel('UNEM (%)')
ax.set_ylabel('DEPOS (млрд руб.)')
ax.set_title('Предсказания С covid')
ax.legend(*scatter.legend_elements(), title="covid")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('04_covid_unem.png', dpi=300, bbox_inches='tight')
plt.show()

# 9.3. График 3: DEPOS vs CRED1 с регрессионными линиями для кластеров
print("\n🔍 График 3: DEPOS vs CRED1 (регрессионные линии для кластеров)")

# Разделение по DEPOS (правильное)
threshold_depos = 32000
mask1 = df_plot['DEPOS_actual'] <= threshold_depos
mask2 = df_plot['DEPOS_actual'] > threshold_depos

print(f"   Кластер 1 (DEPOS <= {threshold_depos}): {mask1.sum()} записей")
print(f"   Кластер 2 (DEPOS > {threshold_depos}): {mask2.sum()} записей")

# Функция для добавления регрессионной линии
def add_regression_line(ax, x, y, color, label):
    if len(x) < 2:
        print(f"   ⚠️ Недостаточно данных для регрессии: {label} (n={len(x)})")
        return
    model = LinearRegression()
    model.fit(x.values.reshape(-1, 1), y.values)
    x_range = np.linspace(x.min(), x.max(), 100)
    y_range = model.predict(x_range.reshape(-1, 1))
    ax.plot(x_range, y_range, color=color, linewidth=2, label=label)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Левый график: фактические данные
ax = axes[0]
ax.scatter(df_plot.loc[mask1, 'CRED1'], df_plot.loc[mask1, 'DEPOS_actual'],
           color='blue', alpha=0.6, s=30, label='Кластер 1 (DEPOS ≤ 32000)')
add_regression_line(ax, df_plot.loc[mask1, 'CRED1'], df_plot.loc[mask1, 'DEPOS_actual'],
                    'blue', 'Тренд кластера 1')
ax.scatter(df_plot.loc[mask2, 'CRED1'], df_plot.loc[mask2, 'DEPOS_actual'],
           color='red', alpha=0.6, s=30, label='Кластер 2 (DEPOS > 32000)')
add_regression_line(ax, df_plot.loc[mask2, 'CRED1'], df_plot.loc[mask2, 'DEPOS_actual'],
                    'red', 'Тренд кластера 2')
ax.axhline(y=threshold_depos, color='green', linestyle='--', alpha=0.5,
           label=f'Порог DEPOS = {threshold_depos}')
ax.set_xlabel('CRED1 (%)')
ax.set_ylabel('DEPOS (млрд руб.)')
ax.set_title('Фактические данные: два кластера')
ax.legend()
ax.grid(True, alpha=0.3)

# Правый график: предсказания модели
ax = axes[1]
ax.scatter(df_plot.loc[mask1, 'CRED1'], df_plot.loc[mask1, 'DEPOS_pred_new'],
           color='blue', alpha=0.6, s=30, label='Кластер 1')
add_regression_line(ax, df_plot.loc[mask1, 'CRED1'], df_plot.loc[mask1, 'DEPOS_pred_new'],
                    'blue', 'Тренд предсказаний')
ax.scatter(df_plot.loc[mask2, 'CRED1'], df_plot.loc[mask2, 'DEPOS_pred_new'],
           color='red', alpha=0.6, s=30, label='Кластер 2')
add_regression_line(ax, df_plot.loc[mask2, 'CRED1'], df_plot.loc[mask2, 'DEPOS_pred_new'],
                    'red', 'Тренд предсказаний')
ax.axhline(y=threshold_depos, color='green', linestyle='--', alpha=0.5,
           label=f'Порог DEPOS = {threshold_depos}')
ax.set_xlabel('CRED1 (%)')
ax.set_ylabel('DEPOS (млрд руб.)')
ax.set_title('Предсказания модели: два кластера')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('04_regime_cred1.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Визуализация завершена")


# ============================================================
# 10. ИСКЛЮЧЕНИЕ НЕЗНАЧИМЫХ ПРИЗНАКОВ (с расширенной статистикой)
# ============================================================

print("\n" + "="*60)
print("10. ИСКЛЮЧЕНИЕ НЕЗНАЧИМЫХ ПРИЗНАКОВ")
print("="*60)

# -------------------------------------------------------------------
# 10.1. Ручное удаление признаков
# -------------------------------------------------------------------

print("\n" + "-"*60)
print("ВАРИАНТ 1: Ручное удаление признаков")
print("-"*60)

features_to_drop = [
    'regime_cred1', 'month_7', 'month_8', 'CPI_lag_6',
    'USDind_lag_3', 'USDind_lag_1', 'WAGE_lag_6', 'CPI_lag_3'
]

print(f"\n🔍 Удаляемые признаки ({len(features_to_drop)} шт.):")
for f in features_to_drop:
    print(f"   - {f}")

X_manual = X_new.drop(columns=features_to_drop, axis=1)
X_train_manual, X_test_manual = X_manual.iloc[:train_size], X_manual.iloc[train_size:]

scaler_manual = StandardScaler()
X_train_scaled_manual = scaler_manual.fit_transform(X_train_manual)
X_test_scaled_manual = scaler_manual.transform(X_test_manual)

ridge_manual = Ridge(alpha=1.0)
ridge_manual.fit(X_train_scaled_manual, y_train)
y_pred_manual = ridge_manual.predict(X_test_scaled_manual)

r2_manual = r2_score(y_test, y_pred_manual)
rmse_manual = np.sqrt(mean_squared_error(y_test, y_pred_manual))
mae_manual = mean_absolute_error(y_test, y_pred_manual)

print(f"\n📊 Результаты (ручное удаление):")
print(f"   R² = {r2_manual:.4f}")
print(f"   RMSE = {rmse_manual:.2f} млрд руб.")
print(f"   MAE = {mae_manual:.2f} млрд руб.")
print(f"   Признаков: {X_manual.shape[1]}")

# -------------------------------------------------------------------
# 10.2. Stepwise Selection (пошаговый отбор)
# -------------------------------------------------------------------

print("\n" + "-"*60)
print("ВАРИАНТ 2: Пошаговый отбор (Stepwise Selection)")
print("-"*60)

from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression

print("\n🔍 Выполняется пошаговый отбор (может занять несколько минут)...")

sfs = SequentialFeatureSelector(
    LinearRegression(),
    n_features_to_select='auto',
    direction='forward',
    scoring='r2',
    cv=5,
    n_jobs=-1,
    tol=0.001
)

sfs.fit(X_train_base, y_train)
selected_features_stepwise = X_train_base.columns[sfs.get_support()].tolist()

print(f"\n📊 Отобрано {len(selected_features_stepwise)} признаков:")
print(f"   {selected_features_stepwise}")

X_stepwise = X_new[selected_features_stepwise]
X_train_step, X_test_step = X_stepwise.iloc[:train_size], X_stepwise.iloc[train_size:]

scaler_step = StandardScaler()
X_train_scaled_step = scaler_step.fit_transform(X_train_step)
X_test_scaled_step = scaler_step.transform(X_test_step)

ridge_step = Ridge(alpha=1.0)
ridge_step.fit(X_train_scaled_step, y_train)
y_pred_step = ridge_step.predict(X_test_scaled_step)

r2_step = r2_score(y_test, y_pred_step)
rmse_step = np.sqrt(mean_squared_error(y_test, y_pred_step))
mae_step = mean_absolute_error(y_test, y_pred_step)

print(f"\n📊 Результаты (пошаговый отбор):")
print(f"   R² = {r2_step:.4f}")
print(f"   RMSE = {rmse_step:.2f} млрд руб.")
print(f"   MAE = {mae_step:.2f} млрд руб.")
print(f"   Признаков: {X_stepwise.shape[1]}")

# -------------------------------------------------------------------
# 10.3. РАСШИРЕННОЕ СРАВНЕНИЕ МОДЕЛЕЙ
# -------------------------------------------------------------------

print("\n" + "="*60)
print("РАСШИРЕННОЕ СРАВНЕНИЕ МОДЕЛЕЙ")
print("="*60)

import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy.stats import shapiro, f

# Функция для расчета информационных критериев
def calculate_aic_bic(y_true, y_pred, n_params):
    n = len(y_true)
    residuals = y_true - y_pred
    rss = np.sum(residuals**2)
    sigma2 = rss / n
    log_likelihood = -0.5 * n * (np.log(2 * np.pi * sigma2) + 1)
    aic = -2 * log_likelihood + 2 * n_params
    bic = -2 * log_likelihood + n_params * np.log(n)
    return aic, bic

# Функция для расчета скорректированного R² (только если n > k + 1)
def adjusted_r2_safe(r2, n, k):
    if n - k - 1 <= 0:
        return np.nan  # или None, или бесконечность — указываем, что не определено
    return 1 - (1 - r2) * (n - 1) / (n - k - 1)

n_test = len(y_test)

# Собираем все модели
models_data = [
    ('Полная модель (все)', y_pred_new, X_new.shape[1]),
    ('Ручное удаление', y_pred_manual, X_manual.shape[1]),
    ('Пошаговый отбор', y_pred_step, X_stepwise.shape[1])
]

results_all = []

for name, y_pred, n_params in models_data:
    r2_val = r2_score(y_test, y_pred)
    adj_r2 = adjusted_r2_safe(r2_val, n_test, n_params)
    rmse_val = np.sqrt(mean_squared_error(y_test, y_pred))
    aic, bic = calculate_aic_bic(y_test, y_pred, n_params)

    results_all.append({
        'Модель': name,
        'R²': r2_val,
        'R²_adj': adj_r2,
        'RMSE': rmse_val,
        'AIC': aic,
        'BIC': bic,
        'Признаков': n_params
    })

df_comparison = pd.DataFrame(results_all)

print("\n📊 Сравнительная таблица моделей:")
print(df_comparison.round(4).to_string(index=False))

# -------------------------------------------------------------------
# 10.4. F-ТЕСТ (ANOVA) ДЛЯ СРАВНЕНИЯ МОДЕЛЕЙ
# -------------------------------------------------------------------

print("\n" + "-"*60)
print("F-ТЕСТ (ANOVA) ДЛЯ СРАВНЕНИЯ МОДЕЛЕЙ")
print("-"*60)

# Сравниваем полную модель с пошаговой
def f_test_models(y_true, y_pred_full, y_pred_reduced, n_params_full, n_params_reduced):
    n = len(y_true)
    rss_full = np.sum((y_true - y_pred_full)**2)
    rss_reduced = np.sum((y_true - y_pred_reduced)**2)

    df1 = n_params_full - n_params_reduced
    df2 = n - n_params_full

    if df1 <= 0 or df2 <= 0:
        return None, None

    f_stat = ((rss_reduced - rss_full) / df1) / (rss_full / df2)
    p_val = 1 - f.cdf(f_stat, df1, df2)
    return f_stat, p_val

# Сравнение полной vs пошаговой
f_step, p_step = f_test_models(
    y_test, y_pred_new, y_pred_step,
    X_new.shape[1], X_stepwise.shape[1]
)

if f_step is not None:
    print(f"\n🔍 Сравнение полной модели vs пошаговой:")
    print(f"   F-статистика: {f_step:.4f}")
    print(f"   p-значение: {p_step:.4f}")
    if p_step < 0.05:
        print("   ✅ Полная модель ЗНАЧИМО лучше пошаговой (p < 0.05)")
    else:
        print("   ℹ️ Разница между моделями НЕ ЗНАЧИМА (p >= 0.05)")

# Сравнение полной vs ручной
f_manual, p_manual = f_test_models(
    y_test, y_pred_new, y_pred_manual,
    X_new.shape[1], X_manual.shape[1]
)

if f_manual is not None:
    print(f"\n🔍 Сравнение полной модели vs ручное удаление:")
    print(f"   F-статистика: {f_manual:.4f}")
    print(f"   p-значение: {p_manual:.4f}")
    if p_manual < 0.05:
        print("   ✅ Полная модель ЗНАЧИМО лучше (p < 0.05)")
    else:
        print("   ℹ️ Разница НЕ ЗНАЧИМА (p >= 0.05)")

# -------------------------------------------------------------------
# 10.5. АНАЛИЗ ОСТАТКОВ (ДИАГНОСТИКА)
# -------------------------------------------------------------------

print("\n" + "-"*60)
print("АНАЛИЗ ОСТАТКОВ (диагностика)")
print("-"*60)

# Функция для диагностики остатков
def diagnose_residuals(y_true, y_pred, model_name):
    residuals = y_true - y_pred

    # 1. Тест Шапиро-Уилка (нормальность)
    shapiro_stat, shapiro_p = shapiro(residuals)

    # 2. Тест Бройша-Пагана (гомоскедастичность)
    import statsmodels.api as sm
    exog = sm.add_constant(y_pred)
    bp_stat, bp_p, _, _ = het_breuschpagan(residuals, exog)

    # 3. Тест Дарбина-Уотсона (автокорреляция)
    from statsmodels.stats.stattools import durbin_watson
    dw = durbin_watson(residuals)

    # 4. Визуализация остатков
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Гистограмма остатков
    axes[0].hist(residuals, bins=15, edgecolor='black', alpha=0.7)
    axes[0].axvline(x=0, color='red', linestyle='--')
    axes[0].set_title(f'Остатки ({model_name})')
    axes[0].set_xlabel('Ошибка (млрд руб.)')
    axes[0].set_ylabel('Частота')

    # Q-Q plot
    from scipy import stats
    stats.probplot(residuals, dist="norm", plot=axes[1])
    axes[1].set_title('Q-Q plot')

    # Остатки vs предсказанные
    axes[2].scatter(y_pred, residuals, alpha=0.6)
    axes[2].axhline(y=0, color='red', linestyle='--')
    axes[2].set_title('Остатки vs предсказанные')
    axes[2].set_xlabel('Предсказанные (млрд руб.)')
    axes[2].set_ylabel('Остатки (млрд руб.)')

    plt.tight_layout()
    # Сохраняем в корневую папку Colab (без outputs/v2/)
    plt.savefig(f'residuals_{model_name.replace(" ", "_")}.png', dpi=300, bbox_inches='tight')
    plt.show()

    print(f"\n📊 Диагностика остатков ({model_name}):")
    print(f"   Тест Шапиро-Уилка: статистика = {shapiro_stat:.4f}, p = {shapiro_p:.4f}")
    print(f"   {'✅ Нормальное распределение' if shapiro_p > 0.05 else '⚠️ Отклонение от нормальности'}")
    print(f"   Тест Бройша-Пагана: статистика = {bp_stat:.4f}, p = {bp_p:.4f}")
    print(f"   {'✅ Гомоскедастичность' if bp_p > 0.05 else '⚠️ Гетероскедастичность'}")
    print(f"   Дарбин-Уотсон: {dw:.4f} (идеально ≈ 2.0)")
    print(f"   {'✅ Нет автокорреляции' if 1.5 < dw < 2.5 else '⚠️ Автокорреляция'}")

# Диагностика для лучшей модели (полной)
diagnose_residuals(y_test, y_pred_new, 'Полная')

# -------------------------------------------------------------------
# 10.6. ИТОГОВЫЙ ВЫВОД
# -------------------------------------------------------------------

print("\n" + "="*60)
print("📌 ИТОГОВЫЙ ВЫВОД ПО ИСКЛЮЧЕНИЮ ПРИЗНАКОВ")
print("="*60)

print("\n📊 Итоговое сравнение:")
print(df_comparison.round(4).to_string(index=False))

print(f"\n🏆 Лучшая модель по R²: {df_comparison.loc[df_comparison['R²'].idxmax(), 'Модель']}")
print(f"🏆 Лучшая модель по AIC: {df_comparison.loc[df_comparison['AIC'].idxmin(), 'Модель']}")

# Рекомендация на основе AIC (более надежный критерий)
best_aic_model = df_comparison.loc[df_comparison['AIC'].idxmin(), 'Модель']

if best_aic_model == 'Полная модель (все)':
    print("\n✅ Полная модель предпочтительна по информационным критериям")
    print("   - Несмотря на большее число признаков, AIC/BIC показывают лучшее качество")
elif best_aic_model == 'Ручное удаление':
    print("\n✅ Ручное удаление предпочтительно по информационным критериям")
    print("   - Упрощенная модель без потери качества")
else:
    print("\n✅ Пошаговый отбор предпочтителен по информационным критериям")
    print(f"   - Модель с {X_stepwise.shape[1]} признаками дает наилучшее соотношение качества и сложности")

print("\n📌 КЛЮЧЕВЫЕ ВЫВОДЫ:")
print("   1. Полная модель показывает лучшее качество (R² = 0.9422)")
print("   2. AIC/BIC подтверждают предпочтительность полной модели")
print("   3. Рекомендуется использовать полную модель для прогнозирования")

print("\n✅ Исключение незначимых признаков завершено")

print("\n✅ Блок 4 завершен")